In [1]:
%load_ext autoreload
%autoreload 2
import scMPRAforge as scm


2025-08-18 12:03:11.526798: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-18 12:03:11.531429: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/code-server/4.91.1/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

In [2]:
import pandas as pd

## Test wald test

In [3]:

#create dask cluster
from dask.distributed import Client, LocalCluster
cluster=LocalCluster(memory_limit='8GB')
client = Client(cluster)

path="/gpfs/gibbs/pi/reilly/tabula_data/shendure"
name="ortho_primordial"

import os
if os.path.isdir(path+"/"+name):
    print("[+] Model found. Loading...")
    primordial=scm.ortho.load(client,path,name)
    shendure=primordial.training_data
else:
    print("[+] Model not found. Creating...")

    #load data
    data_root="/gpfs/gibbs/pi/reilly/tabula_data"
    shendure=scm.scMPRA_data.from_tsv(f"{data_root}/shendure/shendure_counts_grouped.txt")
    
    shendure.set_negative_controls(["minP","noP"])
    shendure.set_reference_cell("Pluripotent")
    shendure.ortho_filter()

    primordial=scm.ortho()
    primordial.criss_cross(client=client,
                       dat=shendure)
    primordial.extract_params(client)
    primordial.save(path,name)

[+] Model found. Loading...


In [4]:
# by-cell-type: test many CREs in NeuroectodermBrain vs the 'reference' negative control
hs_ct = scm.make_by_celltype_hypotheses(
    comparison_cell_type="NeuroectodermBrain",
    counts=shendure,
    comparison_cres="all",          # or a list like ["CRE1","CRE2",...]
    reference_cre="reference",      # this is how you labeled minP/noP
    meta="emvar_screen"
)

# by-CRE: test CRE123 across all cell types vs the baseline cell type
hs_cre = scm.make_by_cre_hypotheses(
    comparison_cre="all",
    counts=shendure,
    comparison_cell_types="all",    # or a list
    reference_cell_type="reference",   # will default to counts.reference_cell_type if set
    meta="cell_specificity"
)

# all CREs within each cell type, vs the 'reference' negative control
hs_all_ct = scm.make_all_by_celltype_hypotheses(
    counts=shendure,
    reference_cre="reference",
    meta="emvar_screen",
)

# all cell types for each CRE, vs the dataset’s baseline cell type
hs_all_cre = scm.make_all_by_cre_hypotheses(
    counts=shendure,
    reference_cell_type="reference",  # will be normalized to 'reference'
    meta="cell_specificity",
)




In [6]:
hs_all_ct.to_dataframe()

,comparison_CRE,comparison_cell_type,reference_CRE,reference_cell_type,meta
0,Bend5_chr4_8175,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen
1,Cdk5r1_chr11_12559,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen
2,Col1a1_chr11_15322,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen
3,Col1a2_chr6_77,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen
4,Igfbp4_chr11_16711,Cardiomyocytes,reference,Cardiomyocytes,emvar_screen
...,...,...,...,...,...
1446,Txndc12_chr4_7973,reference,reference,reference,emvar_screen
1447,Lama1_chr17_7787,reference,reference,reference,emvar_screen
1448,Epas1_chr17_10064,reference,reference,reference,emvar_screen
1449,Btg1_chr10_9578,reference,reference,reference,emvar_screen


In [7]:
hs_all_cre.to_dataframe()

,comparison_CRE,comparison_cell_type,reference_CRE,reference_cell_type,meta
0,Bend5_chr4_8168,NeuroectodermBrain,Bend5_chr4_8168,reference,cell_specificity
1,Bend5_chr4_8168,ExEndodermParietal,Bend5_chr4_8168,reference,cell_specificity
2,Bend5_chr4_8168,EpiblastPrimitiveStreak,Bend5_chr4_8168,reference,cell_specificity
3,Bend5_chr4_8168,SurfaceEctoderm,Bend5_chr4_8168,reference,cell_specificity
4,Bend5_chr4_8170,EpiblastPrimitiveStreak,Bend5_chr4_8170,reference,cell_specificity
...,...,...,...,...,...
1240,ubcP,NeuroectodermRostral,ubcP,reference,cell_specificity
1241,ubcP,ExEndodermParietal,ubcP,reference,cell_specificity
1242,ubcP,Haematoendothelial,ubcP,reference,cell_specificity
1243,ubcP,Cardiomyocytes,ubcP,reference,cell_specificity


In [8]:
# Run the Wald test you already wired up
test_fn = scm.build_wald_test_fn(primordial, shendure, cre_mode="vs_reference")  # or "top_vs_rest"
runner  = scm.HypothesisTester(test_fn=test_fn, test_type_name="wald")


In [9]:
wald_by_cre = runner.run(hs_all_cre).to_dataframe()

2025-08-18 12:05:01.614964: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-08-18 12:05:01.615013: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (r814u03n06.mccleary.ycrc.yale.edu): /proc/driver/nvidia/version does not exist
2025-08-18 12:05:01.621699: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
wald_by_cre

In [22]:
wald_by_ct  = runner.run(hs_all_ct).to_dataframe()

,comparison_CRE,comparison_cell_type,reference_CRE,reference_cell_type,meta,test_statistic,p_value,fold_change,flattened,test_type,bh_p
0,Bend5_chr4_8168,NeuroectodermBrain,Bend5_chr4_8168,referemce,cell_specificity,-1.533897,0.125055,0.439132,False,wald,0.128852
1,Bend5_chr4_8168,ExEndodermParietal,Bend5_chr4_8168,referemce,cell_specificity,-0.505281,0.613361,0.747964,False,wald,0.128852
2,Bend5_chr4_8168,reference,Bend5_chr4_8168,referemce,cell_specificity,0.000000,1.000000,1.000000,False,wald,0.625274
3,Bend5_chr4_8168,EpiblastPrimitiveStreak,Bend5_chr4_8168,referemce,cell_specificity,-0.830511,0.406250,0.600922,False,wald,0.128852
4,Bend5_chr4_8168,SurfaceEctoderm,Bend5_chr4_8168,referemce,cell_specificity,-1.518645,0.128852,0.417336,False,wald,0.128852
